# Download cHL Dataset and Preprocessing

This tutorial will walk you to prepare examples data from the cHL dataset (Shaban et al., MAPS) to use KRONOS. 

### Marker Metadata

Before running KRONOS, we need to define marker metadata. Unlike RGB images, where channels are fixed (red, green, blue) and normalization values are often hardcoded, spatial proteomics (SP) datasets vary in the number and type of channels. Marker names also differ in formatting (e.g., “KI67” might appear as “Ki-67” or “KI-67”) To address this, KRONOS expects a CSV file containing metadata for all markers embedded in the inference data. An initial CSV with 175 markers is provided in [marker_metadata.csv](https://huggingface.co/MahmoodLab/KRONOS/blob/main/marker_metadata.csv).

The marker metadata CSV includes four columns:
- **marker_name**: The name of the marker in uppercase.
- **marker_id**: A unique identifier assigned to the marker in the pretraining dataset.
- **marker_mean**: The mean intensity value of the marker, calculated from a reference dataset (e.g., KRONOS pretraining dataset).
- **marker_std**: The standard deviation intensity value of the marker, also calculated from a reference dataset.

<br/>

### How Marker IDs are assigned to Markers
Marker IDs are assigned as integers from 1 to 512. In the pretrained dataset, nuclear markers are assigned IDs from 1 to 127, while non-nuclear markers receive IDs from 128 to 512. This grouping helps capture high-level similarities between markers of the same type. Within each category, markers are arranged alphabetically, but only even-numbered IDs are assigned to those included in the pretrained dataset. The odd-numbered IDs are intentionally left unassigned, reserved for biologically similar markers that were not part of the pretrained dataset. This approach allows end-users to assign marker IDs from the odd-numbered values, ensuring that any newly added markers remain closely linked to the existing structure while preserving biological relevance.

## Step 1: Download dataset and marker metadata

In [1]:
# Download cHL dataset (MAPS) and marker metadata 
from huggingface_hub import hf_hub_download
import shutil
import os
from utils.codex_dataset_prep import compute_stats

# Set project dir 
project_dir = "./imc_dataset"
target_dir = os.path.join(project_dir, "dataset")

# Download and prepare cHL dataset
# /nfs/turbo/umms-drjieliu/proj/xianzaiHPAP-Spatial/CODEX/hpapdata
stats = compute_stats('/nfs/turbo/umms-drjieliu1/projects/HPAP-Spatial/IMC/hpapdata_comb', os.path.join(project_dir, "dataset"))


local_metadata_source = "../model_assets/marker_metadata.csv" 

if os.path.exists(local_metadata_source):
    shutil.copy(local_metadata_source, os.path.join(target_dir, "marker_metadata.csv"))
    print(f"✅ 成功：已从本地 {local_metadata_source} 拷贝元数据字典。")
else:
    print(f"❌ 错误：在 {local_metadata_source} 找不到文件！请检查你上传的位置。")

# Download marker_metadata.csv 
# cached_file = hf_hub_download(
#     repo_id="MahmoodLab/KRONOS",
#     filename="marker_metadata.csv"
# )
# shutil.copy(cached_file, os.path.join(os.path.join(project_dir, "dataset"), "marker_metadata.csv"))

✅ 成功：已从本地 ../model_assets/marker_metadata.csv 拷贝元数据字典。


## Step 2: Matching pretraining markers with cHL data markers 

The following script maps the marker information (stored in `marker_info.csv`) from the original dataset to those used in the KRONOS pretraining dataset (`marker_metadata.csv`). <br />
It also displays a list of unmatched markers along with suggestions derived from marker name similarity with entries in `marker_metadata.csv`.

In [2]:
from utils import MarkerMetadata
# Define the project directory
project_dir = "./imc_dataset"  # Replace with your actual project directory
# Define paths for the dataset-specific marker info and the pretrained marker metadata files.
marker_info_csv_path = f"{project_dir}/dataset/marker_info.csv"        # Path to the dataset-specific marker info file.
marker_metadata_csv_path = f"{project_dir}/dataset/marker_metadata.csv"  # Path to the pretrained marker metadata file.
top_suggestions = 5  # Number of top suggestions to display for each unmatched marker.

# Create an instance of MarkerMetadata and retrieve the marker metadata.
obj = MarkerMetadata(marker_info_csv_path, marker_metadata_csv_path, top_suggestions)
obj.get_marker_metadata()

# Display the number of markers that do not match the pretrained dataset.
print(f"There are {len(obj.missing_marker_dict)} markers that do not match with the markers in the pretrained dataset.")

# Show the top suggestions based on marker name similarity for each unmatched marker.
print(f"Below are the top {top_suggestions} marker name similarity suggestions for each missing marker:")
display(obj.missing_marker_df)

# Display the dictionary for missing markers, which needs to be manually mapped to a biologically similar marker in marker_metadata.csv.
print("The following dictionary contains missing markers that need to be manually mapped:")
display(obj.missing_marker_dict)

There are 22 markers that do not match with the markers in the pretrained dataset.
Below are the top 5 marker name similarity suggestions for each missing marker:


,Suggestion 1,Suggestion 2,Suggestion 3,Suggestion 4,Suggestion 5
Missing Marker,,,,,
ACTB,MCT,TBET,CDT1,CD1B,ATRX
CA2,CD2,CA9,C4A,BDCA-2,IGA2
CD99,CD69,CD39,CD19,CD49A,CD7
CHGA,CGAS,CA9,C4A,IGA2,IGA1
CK19,CD19,CA9,C1Q,TCF1,MUC1
COL1,CXCL13,C1Q,TCF1,PDL1,MUC1
CPEP,PTEN,ACE2,EPCAM,CLEC9A,CATHEPSIN L
DNA,DAPI,CD1A,VDAC1,EBNA1,CD49A
GCG,CGAS,CS,SIGLEC-9,SIGLEC-7,SIGELC-3


The following dictionary contains missing markers that need to be manually mapped:


{'ACTB': '',
 'CA2': '',
 'CD99': '',
 'CHGA': '',
 'CK19': '',
 'COL1': '',
 'CPEP': '',
 'DNA': '',
 'GCG': '',
 'GHRL': '',
 'HLA-ABC': '',
 'HLA-DR': '',
 'IAPP': '',
 'KRT': '',
 'NES': '',
 'NFKB': '',
 'NKX6-1': '',
 'P16': '',
 'PDX1': '',
 'PPY': '',
 'PS6': '',
 'SST': ''}

## Step 3: Manual marker mapping 

If some markers do not match based on their names, you can manually adjust the mapping. Use the provided suggestions and/or the list of marker names in the marker_metadata.csv file. <br/>
Simply copy the dictionary syntax from the previous step and update the values for the unmatched markers with a valid, biologically similar marker from the suggestions or the `marker_metadata.csv` file.

In [3]:
obj.missing_marker_dict |= {
    "ACTA2": "A-SMA",
    "ECAD": "E-CADHERIN",
    "KRT": "CYTOKERATIN",
    "LAM": "LAMINA",
    "HLA-DR": "HLA_DR",
    "PD-L1": "PDL1",
    "VIM": "VIMENTIN",
    "MMR": "CD206",
}

# Retrieve marker metadata using the updated mapping.
obj.get_marker_metadata_with_mapping()

if len(obj.missing_marker_dict) > 0:
    # Display the count of markers that still do not match the pretrained dataset.
    print(f"There are {len(obj.missing_marker_dict)} markers that still do not match the markers in the pretrained dataset.")

    # Display the dataframe of unmatched markers.
    display(obj.missing_marker_df)

    # Display the dictionary of unmatched markers that require manual mapping.
    display(obj.missing_marker_dict)
else:
    print("All markers have been successfully mapped to the pretrained dataset.")

There are 20 markers that still do not match the markers in the pretrained dataset.


,Suggestion 1,Suggestion 2,Suggestion 3,Suggestion 4,Suggestion 5
Missing Marker,,,,,
ACTB,MCT,TBET,CDT1,CD1B,ATRX
CA2,CD2,CA9,C4A,BDCA-2,IGA2
CD99,CD69,CD39,CD19,CD49A,CD7
CHGA,CGAS,CA9,C4A,IGA2,IGA1
CK19,CD19,CA9,C1Q,TCF1,MUC1
COL1,CXCL13,C1Q,TCF1,PDL1,MUC1
CPEP,PTEN,ACE2,EPCAM,CLEC9A,CATHEPSIN L
DNA,DAPI,CD1A,VDAC1,EBNA1,CD49A
GCG,CGAS,CS,SIGLEC-9,SIGLEC-7,SIGELC-3


{'ACTB': '',
 'CA2': '',
 'CD99': '',
 'CHGA': '',
 'CK19': '',
 'COL1': '',
 'CPEP': '',
 'DNA': '',
 'GCG': '',
 'GHRL': '',
 'HLA-ABC': '',
 'IAPP': '',
 'NES': '',
 'NFKB': '',
 'NKX6-1': '',
 'P16': '',
 'PDX1': '',
 'PPY': '',
 'PS6': '',
 'SST': ''}

## Step 4 (Optional): Manually Set Metadata
If some markers are still unmatched with the pretrained dataset and you can not ignore these marker then you can manually assign their marker ID, mean, and standard deviation values:

- **Marker ID**: Choose an unassigned ID from the range 1–512 in marker_metadata.csv. Ideally, select an ID close to a biologically similar marker.
- **Mean & Std Values**: Calculate these from your dataset for the corresponding markers. Ensure marker intensities are converted to float type and intensities are in range of 0-1 before computing the mean and standard deviation.

In [4]:
new_marker_id_map = {
    # --- Nuclear Markers: 与CODEX共享，使用相同ID ---
    "NKX6-1": 53,   # same as CODEX (close to PAX5=52)
    "PDX1": 57,     # same as CODEX (close to pHH3=56)

    # --- Nuclear Markers: IMC独有 ---
    "DNA": 3,       # nuclear stain, close to DAPI=4
    "NFKB": 47,     # transcription factor, between KI67=46 and P53=50
    "P16": 61,      # cell cycle regulator, close to PROX1=62

    # --- Non-nuclear Markers: 与CODEX共享，使用相同ID ---
    "CHGA": 321,    # same as CODEX
    "CPEP": 341,    # same as CODEX
    "GCG": 323,     # same as CODEX
    "GHRL": 325,    # same as CODEX
    "PPY": 333,     # same as CODEX
    "SST": 335,     # same as CODEX

    # --- Non-nuclear Markers: IMC独有 ---
    "COL1": 313,    # collagen type I (generic), close to COLLAGEN=312; COL1A1=311 in CODEX
    "ACTB": 143,    # cytoskeletal, between B-CATENIN=142 and B-TUBULIN=144
    "CA2": 159,     # carbonic anhydrase, close to CA9=160
    "CD99": 295,    # membrane marker, close to CD8=294
    "CK19": 319,    # cytokeratin 19, between CXCL13=318 and CXCR5=320
    "NES": 345,     # nestin (cytoskeletal), between G6PD=344 and GALECTIN-3=346
    "IAPP": 343,    # islet amyloid polypeptide, between FN1=340 and G6PD=344
    "HLA-ABC": 367, # HLA class I, between HLA_1=366 and HLA_DR=370
    "PS6": 459,     # phospho-S6 (signaling), between RAS=458 and S100=460
}
marker_id_map = {marker: marker_id for marker, marker_id in zip(obj.marker_info['marker_name'], obj.marker_info['marker_id']) if marker_id != 0} | new_marker_id_map
marker_metadata_dict = {marker: {"marker_id": marker_id, "marker_mean": stats.loc[stats['marker_name'] == marker, 'marker_mean'].item(), "marker_std": stats.loc[stats['marker_name'] == marker, 'marker_std'].item()} for marker, marker_id in marker_id_map.items()}

obj.set_marker_metadata(marker_metadata_dict)
if len(obj.missing_marker_dict) > 0:
    print(f"There are {len(obj.missing_marker_dict)} markers that still do not match the markers in the pretrained dataset.")
    display(obj.missing_marker_df)
    display(obj.missing_marker_dict)
else:
    print("All markers now have valid metadata.")
display(obj.marker_info)

All markers now have valid metadata.


,channel_id,marker_name,marker_mean,marker_std,marker_id
0,0,ACTB,11.808868,42.455129,143
1,1,CA2,2.306219,3.685069,159
2,2,CD11B,10.823041,77.481297,180
3,3,CD14,5.071527,259.150305,192
4,4,CD16,1.657781,1.842630,196
5,5,CD163,3.191031,6.717124,200
6,6,CD20,2.379896,112.775620,214
7,7,CD3,2.103989,9.746221,232
8,8,CD31,2.680285,7.777502,236
9,9,CD4,2.372464,9.192493,248


## Step 5: Save Final Dataset Specific Metadata File

In [5]:
output_csv_path = f"{project_dir}/dataset/marker_info_with_metadata.csv"
obj.export_marker_metadata(output_csv_path)
display(obj.marker_info)

Exported marker metadata to ./imc_dataset/dataset/marker_info_with_metadata.csv


,channel_id,marker_name,marker_mean,marker_std,marker_id
0,0,ACTB,11.808868,42.455129,143
1,1,CA2,2.306219,3.685069,159
2,2,CD11B,10.823041,77.481297,180
3,3,CD14,5.071527,259.150305,192
4,4,CD16,1.657781,1.842630,196
5,5,CD163,3.191031,6.717124,200
6,6,CD20,2.379896,112.775620,214
7,7,CD3,2.103989,9.746221,232
8,8,CD31,2.680285,7.777502,236
9,9,CD4,2.372464,9.192493,248
